# Milestone 4: Diagnostic EDA Walkthrough
## "Why Did It Happen?" — Interactive Exploration

**Purpose**: Drill into diagnostic findings using generated evidence tables. Explore customer cohorts, revenue drivers, return patterns, and anomalies interactively.

**Execution**: Run cell-by-cell to explore each diagnostic dimension.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup
OUTPUT_TABLE_DIR = Path('../outputs/tables')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 1e5 else f'{x:.0f}')

print('Setup complete. Loading diagnostic tables...')

## Part 1: Revenue Decomposition by Acquisition Channel
**Question**: Why does Organic Search dominate revenue?

In [ ]:
# Load revenue data
revenue_by_channel = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_revenue_by_acquisition_channel.csv')
display(revenue_by_channel.sort_values('total_revenue', ascending=False))

# Summary statistics
print(f"\nTotal Revenue: ${revenue_by_channel['total_revenue'].sum():,.0f}")
print(f"Channel Concentration (Herfindahl Index): {(revenue_by_channel['revenue_pct']**2).sum():.4f}")
print(f"  -> <0.25 = diversified; >0.25 = concentrated")
print(f"\nTop Channel (Organic): {revenue_by_channel.loc[revenue_by_channel['total_revenue'].idxmax(), 'revenue_pct']:.1%} of total")

In [ ]:
# Visualize channel revenue distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
revenue_by_channel_sorted = revenue_by_channel.sort_values('total_revenue', ascending=False)
colors = sns.color_palette('husl', len(revenue_by_channel_sorted))
ax1.pie(revenue_by_channel_sorted['total_revenue'], labels=revenue_by_channel_sorted['acquisition_channel'], 
        autopct='%1.1f%%', colors=colors)
ax1.set_title('Revenue Distribution by Acquisition Channel')

# Bar chart with avg item value overlay
ax2_twin = ax2.twinx()
bars = ax2.bar(revenue_by_channel_sorted['acquisition_channel'], 
                revenue_by_channel_sorted['total_revenue']/1e9, color=colors, alpha=0.7, label='Revenue ($B)')
line = ax2_twin.plot(revenue_by_channel_sorted['acquisition_channel'], 
                      revenue_by_channel_sorted['avg_item_value'], 'r-o', linewidth=2, markersize=8, label='Avg Item Value')
ax2.set_ylabel('Revenue (Billions $)', fontsize=11)
ax2_twin.set_ylabel('Avg Item Value ($)', fontsize=11, color='red')
ax2.set_xlabel('Acquisition Channel', fontsize=11)
ax2.set_title('Revenue & Item Value by Channel')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n✓ Key Insight: Avg item values are uniform ($21.7K–$22.1K) → Revenue dominance is VOLUME-driven, not pricing-driven")

In [ ]:
# Monthly revenue trends by channel
revenue_monthly = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_revenue_decomposition_monthly.csv')
revenue_monthly['YearMonth'] = pd.to_datetime(revenue_monthly['YearMonth'])

plt.figure(figsize=(14, 6))
for channel in revenue_monthly['acquisition_channel'].unique():
    data = revenue_monthly[revenue_monthly['acquisition_channel'] == channel].sort_values('YearMonth')
    plt.plot(data['YearMonth'], data['revenue']/1e9, marker='o', label=channel, linewidth=2, markersize=4)

plt.xlabel('Month', fontsize=11)
plt.ylabel('Revenue (Billions $)', fontsize=11)
plt.title('Monthly Revenue Trends by Acquisition Channel')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Observation: Channel revenue proportions remain stable over time (mix effect is structural)")

## Part 2: Customer Cohort Analysis (Lifetime Value & Repeat Patterns)
**Question**: Why is customer lifetime value uniform across channels?

In [ ]:
# Load cohort data
cohort = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_customer_cohort_by_channel.csv')
display(cohort.sort_values('avg_ltv', ascending=False))

# Statistics
print(f"\nLTV Statistics Across Channels:")
print(f"  Min LTV: ${cohort['avg_ltv'].min():,.0f}")
print(f"  Max LTV: ${cohort['avg_ltv'].max():,.0f}")
print(f"  Range: ${cohort['avg_ltv'].max() - cohort['avg_ltv'].min():,.0f}")
print(f"  Coefficient of Variation: {cohort['avg_ltv'].std() / cohort['avg_ltv'].mean():.2%}")
print(f"\nRepeat Rate Across Channels:")
print(f"  Min: {cohort['repeat_rate'].min():.2%}")
print(f"  Max: {cohort['repeat_rate'].max():.2%}")
print(f"  Variance: {cohort['repeat_rate'].std():.4f}")
print(f"\n✓ Interpretation: <2% LTV variance + uniform 55% repeat rate = customer quality is channel-AGNOSTIC")

In [ ]:
# Visualize cohort metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
cohort_sorted = cohort.sort_values('avg_ltv', ascending=False)
colors = sns.color_palette('Set2', len(cohort_sorted))

# Avg LTV
axes[0, 0].barh(cohort_sorted['acquisition_channel'], cohort_sorted['avg_ltv']/1e3, color=colors)
axes[0, 0].set_xlabel('Avg LTV ($K)', fontsize=11)
axes[0, 0].set_title('Average Lifetime Value by Channel')
for i, v in enumerate(cohort_sorted['avg_ltv']/1e3):
    axes[0, 0].text(v + 2, i, f'${v:.0f}K', va='center', fontsize=9)

# Median LTV (distribution proxy)
axes[0, 1].barh(cohort_sorted['acquisition_channel'], cohort_sorted['median_ltv']/1e3, color=colors, alpha=0.7)
axes[0, 1].set_xlabel('Median LTV ($K)', fontsize=11)
axes[0, 1].set_title('Median Lifetime Value by Channel')

# Repeat rate
axes[1, 0].barh(cohort_sorted['acquisition_channel'], cohort_sorted['repeat_rate'], color=colors)
axes[1, 0].set_xlabel('Repeat Purchase Rate', fontsize=11)
axes[1, 0].set_title('Repeat Purchase Rate by Channel')
axes[1, 0].set_xlim([0.50, 0.58])
for i, v in enumerate(cohort_sorted['repeat_rate']):
    axes[1, 0].text(v + 0.001, i, f'{v:.1%}', va='center', fontsize=9)

# Avg orders per customer
axes[1, 1].barh(cohort_sorted['acquisition_channel'], cohort_sorted['avg_orders'], color=colors)
axes[1, 1].set_xlabel('Avg Orders per Customer', fontsize=11)
axes[1, 1].set_title('Purchase Frequency by Channel')
for i, v in enumerate(cohort_sorted['avg_orders']):
    axes[1, 1].text(v + 0.05, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n✓ Key Insight: LTV and repeat rates are UNIFORM across channels → acquisition channel does NOT predict customer value")

## Part 3: Return & Quality Drivers
**Question**: Why are return rates similar across channels but vary by category?

In [ ]:
# Load return data
returns_category = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_return_rate_by_category.csv')
returns_channel = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_return_rate_by_channel.csv')
return_reasons = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_return_reasons.csv')

print("Return Rates by Category:")
display(returns_category.sort_values('return_rate', ascending=False))

print("\nReturn Rates by Acquisition Channel:")
display(returns_channel.sort_values('return_rate', ascending=False))

print("\nReturn Reasons (Top 10):")
display(return_reasons.head(10))

In [ ]:
# Visualize return patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By category
returns_category_sorted = returns_category.sort_values('return_rate', ascending=False)
colors1 = ['#ff7f0e' if x > returns_category_sorted['return_rate'].mean() else '#1f77b4' 
           for x in returns_category_sorted['return_rate']]
bars1 = axes[0].bar(returns_category_sorted['category'], returns_category_sorted['return_rate']*100, color=colors1)
axes[0].axhline(y=returns_category_sorted['return_rate'].mean()*100, color='red', linestyle='--', linewidth=2, label='Avg')
axes[0].set_ylabel('Return Rate (%)', fontsize=11)
axes[0].set_xlabel('Product Category', fontsize=11)
axes[0].set_title('Return Rate by Product Category')
axes[0].legend()
for i, (cat, rate) in enumerate(zip(returns_category_sorted['category'], returns_category_sorted['return_rate'])):
    axes[0].text(i, rate*100 + 0.1, f'{rate:.2%}', ha='center', fontsize=10)

# By channel
returns_channel_sorted = returns_channel.sort_values('return_rate', ascending=False)
colors2 = ['#1f77b4'] * len(returns_channel_sorted)  # All same color (no outliers)
axes[1].bar(returns_channel_sorted['acquisition_channel'], returns_channel_sorted['return_rate']*100, color=colors2)
axes[1].axhline(y=returns_channel_sorted['return_rate'].mean()*100, color='red', linestyle='--', linewidth=2, label='Avg')
axes[1].set_ylabel('Return Rate (%)', fontsize=11)
axes[1].set_xlabel('Acquisition Channel', fontsize=11)
axes[1].set_title('Return Rate by Acquisition Channel')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
for i, (chan, rate) in enumerate(zip(returns_channel_sorted['acquisition_channel'], returns_channel_sorted['return_rate'])):
    axes[1].text(i, rate*100 + 0.05, f'{rate:.2%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n✓ Key Insight: Return rate VARIANCE by category (0.26%) but NO variance by channel → product quality, not customer channel, drives returns")

In [ ]:
# Return reasons deep dive
fig, ax = plt.subplots(figsize=(12, 6))
return_reasons_sorted = return_reasons.sort_values('count', ascending=True).tail(15)
bars = ax.barh(return_reasons_sorted['return_reason'], return_reasons_sorted['count'], color='steelblue')
ax.set_xlabel('Return Count', fontsize=11)
ax.set_title('Top 15 Return Reasons')
for i, (reason, count, pct) in enumerate(zip(return_reasons_sorted['return_reason'], 
                                               return_reasons_sorted['count'], 
                                               return_reasons_sorted['percentage'])):
    ax.text(count + 200, i, f"{count:,} ({pct:.1%})", va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nTop 3 Return Reasons:")
for idx, row in return_reasons.head(3).iterrows():
    print(f"  {row['return_reason']}: {row['count']:,} ({row['percentage']:.1%})")
print(f"\n✓ Actionable: 'Wrong Size' is top reason → Improve size guides and product descriptions")

## Part 4: Traffic Funnel & Conversion Analysis
**Question**: Why is web traffic correlation to revenue weak?

In [ ]:
# Load traffic data
traffic_summary = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_traffic_funnel_summary.csv')
traffic_monthly = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_traffic_funnel_monthly.csv')
traffic_monthly['YearMonth'] = pd.to_datetime(traffic_monthly['YearMonth'])

print("Traffic Funnel Health by Source:")
display(traffic_summary.sort_values('total_sessions', ascending=False))

print("\nConversion Rate Statistics:")
print(f"  Avg Conversion Rate: {traffic_summary['avg_conversion_rate'].mean():.3%}")
print(f"  Range: {traffic_summary['avg_conversion_rate'].min():.3%} - {traffic_summary['avg_conversion_rate'].max():.3%}")
print(f"  Std Dev: {traffic_summary['avg_conversion_rate'].std():.3%}")

In [ ]:
# Visualize traffic funnel
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

traffic_summary_sorted = traffic_summary.sort_values('total_sessions', ascending=False)

# Sessions by source
axes[0].barh(traffic_summary_sorted['traffic_source'], traffic_summary_sorted['total_sessions']/1e6, color='steelblue')
axes[0].set_xlabel('Total Sessions (Millions)', fontsize=11)
axes[0].set_title('Web Traffic Volume by Source')

# Conversion rate by source
colors = ['#2ecc71' if x > traffic_summary_sorted['avg_conversion_rate'].mean() else '#e74c3c' 
          for x in traffic_summary_sorted['avg_conversion_rate']]
axes[1].barh(traffic_summary_sorted['traffic_source'], traffic_summary_sorted['avg_conversion_rate']*100, color=colors)
axes[1].axvline(x=traffic_summary_sorted['avg_conversion_rate'].mean()*100, color='blue', linestyle='--', linewidth=2, label='Avg')
axes[1].set_xlabel('Conversion Rate (%)', fontsize=11)
axes[1].set_title('Conversion Efficiency by Traffic Source')
axes[1].legend()
for i, (source, conv) in enumerate(zip(traffic_summary_sorted['traffic_source'], traffic_summary_sorted['avg_conversion_rate'])):
    axes[1].text(conv*100 + 0.02, i, f'{conv:.2%}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n✓ Observation: Conversion rates are STABLE across sources (1.9–2.2%) → traffic quality is consistent")

In [ ]:
# Trend monthly conversions
fig, ax = plt.subplots(figsize=(14, 6))

for source in traffic_monthly['traffic_source'].unique():
    data = traffic_monthly[traffic_monthly['traffic_source'] == source].sort_values('YearMonth')
    ax.plot(data['YearMonth'], data['avg_conversion_rate']*100, marker='o', label=source, linewidth=2, markersize=4)

ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Conversion Rate (%)', fontsize=11)
ax.set_title('Monthly Conversion Rate Trends by Traffic Source')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Observation: Conversion rates remain stable over time (no degradation) → funnel is healthy")

## Part 5: Anomaly Detection & Investigation
**Question**: What drives revenue anomalies?

In [ ]:
# Load anomaly data
anomalies = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_anomalies_dual_detection.csv')
anomalies['Date'] = pd.to_datetime(anomalies['Date'])

print(f"Total Anomalies Detected: {len(anomalies)}")
print(f"\nAnomaly Classification:")
print(anomalies['anomaly_type'].value_counts())
print(f"\nTop 10 High-Revenue Anomalies:")
display(anomalies.nlargest(10, 'Revenue')[['Date', 'Revenue', 'z_score', 'anomaly_type']])

In [ ]:
# Load sales data for context
import sys
sys.path.append('../')
sales = pd.read_csv('../data/sales.csv', parse_dates=['Date'])

# Plot revenue timeline with anomalies highlighted
fig, ax = plt.subplots(figsize=(16, 6))

# Base revenue
ax.plot(sales['Date'], sales['Revenue']/1e6, color='steelblue', linewidth=1, alpha=0.7, label='Daily Revenue')

# Anomalies by type
both = anomalies[anomalies['anomaly_type'] == 'both_methods']
stat_only = anomalies[anomalies['anomaly_type'] == 'statistical_only']
heur_only = anomalies[anomalies['anomaly_type'] == 'heuristic_only']

ax.scatter(both['Date'], both['Revenue']/1e6, color='red', s=100, marker='o', label='Both Methods', zorder=5, alpha=0.8)
ax.scatter(stat_only['Date'], stat_only['Revenue']/1e6, color='orange', s=80, marker='^', label='Statistical Only', zorder=4, alpha=0.6)
ax.scatter(heur_only['Date'], heur_only['Revenue']/1e6, color='yellow', s=80, marker='s', label='Heuristic Only', zorder=4, alpha=0.6)

ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Revenue (Millions $)', fontsize=11)
ax.set_title('Revenue Timeline with Detected Anomalies (Dual Method Cross-Validation)')
ax.legend(loc='best', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✓ Observation: High-revenue anomalies (red dots) cluster seasonally → likely promotional events, not system failures")
print(f"✓ Implication: Anomalies are EXPECTED business behavior; incorporate promotional calendar into forecasting models")

## Part 6: Correlation Analysis
**Question**: What metrics drive revenue?

In [ ]:
# Load correlation data
corr_matrix = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_correlation_matrix.csv', index_col=0)
revenue_corr = pd.read_csv(OUTPUT_TABLE_DIR / 'm4_revenue_correlations.csv')

print("Revenue Correlations with Key Metrics:")
display(revenue_corr)

print(f"\nInterpretation:")
print(f"  COGS (r=0.976): VERY STRONG → revenue is inventory-driven")
print(f"  Returns (r=0.415): MODERATE → some relationship with quality/volume")
print(f"  Sessions (r=0.283): WEAK → traffic is NOT constraining")
print(f"  Bounce Rate (r≈0.0): NO correlation → traffic quality is stable")

In [ ]:
# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0, square=True, 
            cbar_kws={'label': 'Correlation Coefficient'}, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix: Daily Metrics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Key Insight: COGS dominates (r=0.976), suggesting revenue follows product availability, not demand signals (traffic)")

## Summary: Diagnostic Findings

**Why Does Revenue Vary?**

1. **Acquisition Channel Mix** (30% of revenue from Organic) → Volume-driven, not pricing
2. **Inventory Availability** (95% COGS correlation) → Supply-constrained, not demand-constrained
3. **Promotional Calendar** (Anomalies cluster seasonally) → Events drive spikes, not organic growth
4. **Customer Consistency** (Uniform 55% repeat rate) → Business model drives retention, not channel
5. **Product Quality** (Category-driven returns, not channel-driven) → Focus on product improvements

**Why Traffic Correlation Is Weak?**
- Traffic is not a bottleneck (consistent 2% conversion)
- Revenue is constrained by inventory, not by visitor interest
- Promotional events drive both sales and inventory simultaneously

---

**Next Layer: Predictive EDA (Milestone 5)**
- Forecast revenue using promotional calendar + inventory levels (primary drivers)
- Time-series decomposition (trend, seasonality, promotional spike, residual)
- Scenario analysis (e.g., "What if we increase inventory?", "What if we shift promotional calendar?")

In [ ]:
print("\n" + "="*80)
print("MILESTONE 4 DIAGNOSTIC EDA COMPLETE")
print("="*80)
print("\nAll diagnostic evidence tables available in: outputs/tables/m4_*.csv")
print("Formal report: docs/competition/first-round/planning/m4-diagnostic-eda-report.md")
print("\nReady for Milestone 5 (Predictive & Prescriptive EDA)")